In [1]:
import os
import time
import random
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

BASE_DIR = "output"
CACHE_DIR = os.path.join(BASE_DIR, "cache")
FIG_DIR = os.path.join(BASE_DIR, "figures")
TABLE_DIR = os.path.join(BASE_DIR, "tables")
FINAL_DIR = os.path.join(BASE_DIR, "final")

for d in [BASE_DIR, CACHE_DIR, FIG_DIR, TABLE_DIR, FINAL_DIR]:
    os.makedirs(d, exist_ok=True)

def sleep_polite(base=1.5, jitter=1.0):
    time.sleep(base + random.random() * jitter)

rets = pd.read_csv(os.path.join(FINAL_DIR, "daily_returns.csv"), index_col=0, parse_dates=True)

mkt = rets["SPY"].copy()
stock_rets = rets.drop(columns=["SPY"])

def get_earnings_dates_for_ticker(ticker, limit=12):
    """
    Pulls earnings dates via yfinance and caches per ticker to avoid repeated calls.
    """
    cache_path = os.path.join(CACHE_DIR, f"earnings_dates_{ticker}.csv")
    if os.path.exists(cache_path):
        return pd.read_csv(cache_path, parse_dates=["earnings_datetime"])

    t = yf.Ticker(ticker)
    try:
        ed = t.get_earnings_dates(limit=limit)
        if ed is None or len(ed) == 0:
            df = pd.DataFrame(columns=["earnings_datetime"])
        else:
            df = ed.reset_index().rename(columns={"Earnings Date": "earnings_datetime"})
            df = df[["earnings_datetime"]].dropna()
        df.to_csv(cache_path, index=False)
        sleep_polite(base=2.0, jitter=1.5)
        return df
    except Exception:
        df = pd.DataFrame(columns=["earnings_datetime"])
        df.to_csv(cache_path, index=False)
        sleep_polite(base=4.0, jitter=2.5)
        return df

def nearest_trading_day(ts, trading_index):
    if pd.isna(ts):
        return None
    dt = pd.Timestamp(ts).tz_localize(None)
    candidates = trading_index[trading_index >= dt.normalize()]
    if len(candidates) == 0:
        return None
    return candidates[0]

def compute_car(ticker, event_day, tau1, tau2):
    idx = stock_rets.index
    if event_day not in idx:
        return np.nan
    pos = idx.get_loc(event_day)
    start = pos + tau1
    end = pos + tau2
    if start < 0 or end >= len(idx):
        return np.nan
    days = idx[start:end+1]
    ar = stock_rets.loc[days, ticker] - mkt.loc[days]
    return float(np.nansum(ar.values))

def build_event_panel(tickers, max_events_per_ticker=12):
    events = []
    trading_days = stock_rets.index

    for j, tic in enumerate(tickers):
        df_ed = get_earnings_dates_for_ticker(tic, limit=max_events_per_ticker)
        if df_ed.empty:
            continue

        for ts in df_ed["earnings_datetime"].tolist():
            event_day = nearest_trading_day(ts, trading_days)
            if event_day is None:
                continue
            events.append({"ticker": tic, "earnings_datetime": ts, "event_day": event_day})

        if (j + 1) % 30 == 0:
            sleep_polite(base=3.0, jitter=2.0)

    return pd.DataFrame(events)

tickers = stock_rets.columns.tolist()

# Start smaller if you want to reduce API strain
# tickers = tickers[:120]

event_panel = build_event_panel(tickers, max_events_per_ticker=12)

# Surprise proxy CAR(0,1), drift outcome CAR(2,21)
event_panel["surprise_car_0_1"] = event_panel.apply(
    lambda r: compute_car(r["ticker"], r["event_day"], 0, 1),
    axis=1
)
event_panel["drift_car_2_21"] = event_panel.apply(
    lambda r: compute_car(r["ticker"], r["event_day"], 2, 21),
    axis=1
)

# Earnings congestion based on events within sample universe
daily_counts = event_panel.groupby("event_day")["ticker"].count().rename("earnings_count")
event_panel = event_panel.merge(daily_counts, left_on="event_day", right_index=True, how="left")

med = float(event_panel["earnings_count"].median()) if len(event_panel) else 0.0
event_panel["high_distraction"] = (event_panel["earnings_count"] > med).astype(int)

# Basic cleaning
event_panel = event_panel.dropna(subset=["surprise_car_0_1", "drift_car_2_21"])

event_path = os.path.join(FINAL_DIR, "event_panel_yf.csv")
event_panel.to_csv(event_path, index=False)
print(f"Saved event panel to {event_path}")
print("Events:", len(event_panel))

# Report plots
plt.figure()
event_panel["surprise_car_0_1"].hist(bins=60)
plt.title("Distribution of Earnings Surprise Proxy (CAR 0 to 1)")
plt.xlabel("Surprise Proxy")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "surprise_proxy_distribution.png"), dpi=200)
plt.close()

plt.figure()
event_panel["drift_car_2_21"].hist(bins=60)
plt.title("Distribution of Post-Earnings Drift Outcome (CAR 2 to 21)")
plt.xlabel("Drift Outcome")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "drift_distribution.png"), dpi=200)
plt.close()

# Earnings congestion plot
counts_ts = daily_counts.sort_index()
plt.figure()
counts_ts.plot()
plt.title("Earnings Congestion Within Sample Universe")
plt.xlabel("Date")
plt.ylabel("Number of Earnings Announcements")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "earnings_congestion_over_time.png"), dpi=200)
plt.close()

# Table for report
event_panel[["ticker","event_day","surprise_car_0_1","drift_car_2_21","earnings_count","high_distraction"]].head(50)\
    .to_csv(os.path.join(TABLE_DIR, "event_panel_preview.csv"), index=False)

print("Saved preview table to output/tables/event_panel_preview.csv")

Saved event panel to output/final/event_panel_yf.csv
Events: 382
Saved preview table to output/tables/event_panel_preview.csv
